## Import & Load Data

In [2]:
import pandas as pd

train = pd.read_csv("churn-bigml-80.csv")
test = pd.read_csv("churn-bigml-20.csv")

train.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


## Understand Data

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   object 
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   object 
 4   Voice mail plan         2666 non-null   object 
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total day charge        2666 non-null   float64
 9   Total eve minutes       2666 non-null   float64
 10  Total eve calls         2666 non-null   int64  
 11  Total eve charge        2666 non-null   float64
 12  Total night minutes     2666 non-null   float64
 13  Total night calls       2666 non-null   int64  
 14  Total night charge      2666 non-null   

In [4]:
train.isnull().sum()

State                     0
Account length            0
Area code                 0
International plan        0
Voice mail plan           0
Number vmail messages     0
Total day minutes         0
Total day calls           0
Total day charge          0
Total eve minutes         0
Total eve calls           0
Total eve charge          0
Total night minutes       0
Total night calls         0
Total night charge        0
Total intl minutes        0
Total intl calls          0
Total intl charge         0
Customer service calls    0
Churn                     0
dtype: int64

In [5]:
train['Churn'].value_counts()

Churn
False    2278
True      388
Name: count, dtype: int64

## Data Cleaning

In [6]:
train = train.drop(['State'], axis=1)
test = test.drop(['State'], axis=1)

In [7]:
for col in ['International plan', 'Voice mail plan']:
    train[col] = train[col].map({'Yes':1, 'No':0})
    test[col] = test[col].map({'Yes':1, 'No':0})

In [8]:
train['Churn'] = train['Churn'].astype(int)
test['Churn'] = test['Churn'].astype(int)

In [9]:
train.dtypes
train.isnull().sum()

Account length            0
Area code                 0
International plan        0
Voice mail plan           0
Number vmail messages     0
Total day minutes         0
Total day calls           0
Total day charge          0
Total eve minutes         0
Total eve calls           0
Total eve charge          0
Total night minutes       0
Total night calls         0
Total night charge        0
Total intl minutes        0
Total intl calls          0
Total intl charge         0
Customer service calls    0
Churn                     0
dtype: int64

## Define Features & Target

In [11]:
X_train = train.drop('Churn', axis=1)
y_train = train['Churn']

X_test = test.drop('Churn', axis=1)
y_test = test['Churn']

### Feature scaling was applied to improve the performance of Logistic Regression, as it is sensitive to the scale of input features.

## Train Models

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=2000)

In [19]:
# Random Forest
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

## Make Predictions

In [20]:
y_pred_lr = lr.predict(X_test_scaled)

In [21]:
y_pred_rf = rf.predict(X_test)

## Evaluate Models

In [22]:
from sklearn.metrics import accuracy_score, classification_report

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nLogistic Report:\n", classification_report(y_test, y_pred_lr))
print("\nRandom Forest Report:\n", classification_report(y_test, y_pred_rf))

Logistic Regression Accuracy: 0.8530734632683659
Random Forest Accuracy: 0.9445277361319341

Logistic Report:
               precision    recall  f1-score   support

           0       0.88      0.97      0.92       572
           1       0.46      0.18      0.26        95

    accuracy                           0.85       667
   macro avg       0.67      0.57      0.59       667
weighted avg       0.82      0.85      0.82       667


Random Forest Report:
               precision    recall  f1-score   support

           0       0.95      0.99      0.97       572
           1       0.91      0.67      0.78        95

    accuracy                           0.94       667
   macro avg       0.93      0.83      0.87       667
weighted avg       0.94      0.94      0.94       667



## Model Performance

### Two models were trained: Logistic Regression and Random Forest.
### The Random Forest model performed better due to its ability to capture complex patterns in the data.

In [23]:
import pandas as pd

feature_importance = pd.Series(rf.feature_importances_, index=X_train.columns)
feature_importance.sort_values(ascending=False)

Total day minutes         0.138338
Total day charge          0.125838
Customer service calls    0.114573
International plan        0.088281
Total eve charge          0.066224
Total eve minutes         0.065635
Total intl calls          0.055894
Total intl charge         0.046005
Total intl minutes        0.044318
Total night minutes       0.040632
Total night charge        0.036454
Account length            0.032449
Total night calls         0.030628
Total day calls           0.030399
Total eve calls           0.028267
Number vmail messages     0.027955
Voice mail plan           0.019245
Area code                 0.008865
dtype: float64

### The Random Forest model highlights that features such as customer service calls, total charges, and international plan usage are among the most important predictors of churn.

### Key Insights

### The analysis revealed that customer behavior plays a major role in churn prediction.

### Customers with higher customer service calls were more likely to churn, indicating possible dissatisfaction.

### Additionally, customers subscribed to international plans showed different churn patterns, suggesting that service type influences retention.

### Overall, usage patterns and customer interactions are strong indicators of whether a customer will leave.